In [35]:
# Problem 2

In [36]:
# Task 1: Implementing Mini Block Cipher with key size 16 bit and block size 16 bit:

In [37]:
# define S-Box and Inverse S-Box

import random

sbox = {
0x0:0x9, 0x1:0x4, 0x2:0xA, 0x3:0xB,
0x4:0xD, 0x5:0x1, 0x6:0x8, 0x7:0x5,
0x8:0x6, 0x9:0x2, 0xA:0x0, 0xB:0x3,
0xC:0xC, 0xD:0xE, 0xE:0xF, 0xF:0x7
}

inv_sbox = {v:k for k,v in sbox.items()}

In [38]:
# Define substitute and inverse substitute functions

def substitute_state(state):

    result = 0

    for i in range(4):
        nibble = (state >> (i*4)) & 0xF
        result |= sbox[nibble] << (i*4)

    return result


def inverse_substitute_state(state):

    result = 0

    for i in range(4):
        nibble = (state >> (i*4)) & 0xF
        result |= inv_sbox[nibble] << (i*4)

    return result

In [39]:
# Shift Op

def shift(state):

    left = (state >> 8) & 0xFF
    right = state & 0xFF

    return (right << 8) | left

In [40]:
# Add Round Key Op

def add_round_key(state, key):

    return state ^ key

In [41]:
# Mix and Inverse Mix functions

def mix(state):

    return ((state << 4) | (state >> 12)) & 0xFFFF


def inverse_mix(state):

    return ((state >> 4) | (state << 12)) & 0xFFFF

In [42]:
# Key expansion

def key_expansion(key):

    K1 = key >> 8
    K2 = key & 0xFF

    return K1, K2

In [43]:
# Round 1 Encrypt

def encrypt_round1(state, K1):

    state = substitute_state(state)
    state = shift(state)
    state = mix(state)
    state = add_round_key(state, K1)

    return state

In [44]:
# Round 2 Encrypt

def encrypt_round2(state, K2):

    state = substitute_state(state)
    state = shift(state)
    state = add_round_key(state, K2)

    return state

In [45]:
# Full Encrypt

def encrypt(plaintext, key):

    K1, K2 = key_expansion(key)

    state = encrypt_round1(plaintext, K1)
    ciphertext = encrypt_round2(state, K2)

    return ciphertext

In [46]:
# Decrypt 

def decrypt_round2(state, K2):

    state = add_round_key(state, K2)
    state = shift(state)
    state = inverse_substitute_state(state)

    return state


def decrypt_round1(state, K1):

    state = add_round_key(state, K1)
    state = inverse_mix(state)
    state = shift(state)
    state = inverse_substitute_state(state)

    return state

In [47]:
# Plaintext Ciphertext Pairs

key = random.randint(0,65535)

pairs = []

for i in range(10):

    P = random.randint(0,65535)
    C = encrypt(P,key)

    pairs.append((P,C))

print("Key used:", key)
print("Plaintext-Ciphertext pairs:")
print(pairs)

Key used: 19858
Plaintext-Ciphertext pairs:
[(39538, 54936), (10135, 18306), (55082, 17677), (53729, 39501), (61700, 40311), (11548, 594), (8126, 49135), (16856, 39708), (32165, 3795), (41804, 31067)]


In [48]:
# Task 2 - MITM Attack

In [49]:
#Table Build

P,C = pairs[0]

table = {}

for K1 in range(256):

    X = encrypt_round1(P,K1)

    if X not in table:
        table[X] = []

    table[X].append(K1)

In [50]:
# MITM Match

matches = []

for K2 in range(256):

    X2 = decrypt_round2(C,K2)

    if X2 in table:

        for K1 in table[X2]:

            matches.append((K1,K2))

print("Candidate key pairs:", matches)

Candidate key pairs: [(77, 146)]


In [51]:
# verify keys

for K1,K2 in matches:

    valid = True

    for P,C in pairs:

        if encrypt_round2(encrypt_round1(P,K1),K2) != C:

            valid = False
            break

    if valid:

        print("Recovered keys:",K1,K2)

Recovered keys: 77 146


In [52]:
# recovered keys

recovered_key = None

for K1, K2 in matches:

    valid = True

    for P, C in pairs:

        if encrypt_round2(encrypt_round1(P, K1), K2) != C:
            valid = False
            break

    if valid:

        recovered_key = (K1 << 8) | K2
        print("Recovered keys:")
        print("K1 =", K1)
        print("K2 =", K2)
        print("Recovered 16-bit key =", recovered_key)

Recovered keys:
K1 = 77
K2 = 146
Recovered 16-bit key = 19858


In [53]:
#MITM Summary

print("\n--- MITM Attack Summary ---")

print("Original key:", key)
print("Recovered key:", recovered_key)

if recovered_key == key:
    print("MITM attack successful: The original key was recovered.")
else:
    print("MITM attack failed: The recovered key does not match the original key.")

print("\nAttack Complexity:")
print("Brute-force attack complexity: 2^32 key combinations")
print("MITM attack complexity: approximately 2^16 + 2^16 operations")


--- MITM Attack Summary ---
Original key: 19858
Recovered key: 19858
MITM attack successful: The original key was recovered.

Attack Complexity:
Brute-force attack complexity: 2^32 key combinations
MITM attack complexity: approximately 2^16 + 2^16 operations


In [54]:
#Task 3

In [55]:
# (a) What is the key space for the mini block cipher?

In [56]:
# solution: The mini block cipher uses a 16-bit key. 
# The size of the key space is therefore: Key space = 2^16 = 65536 possible keys. 
# An attacker performing a brute-force attack would need to try up to 65,536 keys to find the correct one.

In [57]:
# (b) Image the mini block cipher is executed twice to generate a cipher text. It is called double mini cipher block. We need a key in 32 bits. The first 16 to the first mini block cipher, the remaining 16 to the second mini block cipher. The meet in the middle attack is to match the state for the first encryption of mini block cipher and the second decryption mini block. How many operations are needed to such attack?

In [58]:
# Solution: For double mini cipher : Plaintext -> Encrypt(k1) -> encrypt (k2) -> ciphertext. 
# K1=16 Bits, K2=16 Bits. Total key size = 32 bits.
# So the total number of key spaces become 2^32 possible keys.

# For MITM attack, instead of testing all key combinations, we split it into 2 parts. 
# 2^16 for encryptions and 2^ 16 for decryptions which is 2 * 2^16 ~= 2^ 17 operations ~= 131,072 operations

In [59]:
# (c) If we do exhaustive key search for the double mini block cipher, how many operations are needed?

In [60]:
# Solution: 2^32 = 4,294,967,296 operations are required

In [61]:
#  (d) What is the trade off in this attack?

In [62]:
# Solution: The Meet-in-the-Middle attack significantly reduces the computational complexity of breaking a double encryption scheme from  2^32 operations to approximately 2^17 operations by storing intermediate encryption results, at the cost of increased memory usage.

In [63]:
#Task 4: Discuss how we can improve the mini block cipher.

In [84]:
# Solution: he mini block cipher is vulnerable to Meet-in-the-Middle attacks because it lacks an initial AddRoundKey step and has only two rounds of encryption. S-AES improves security by adding an initial key mixing operation and using a more structured design.
# To improve the mini block cipher:
# 1. Add an initial AddRoundKey operation
# 2. Increase the number of rounds
# 3. Use a stronger key expansion algorithm
# 4. Improve the mixing operation
# These changes would significantly increase the cipher’s resistance to cryptographic attacks.